# Dent-FINE 有效模块代码提取与 Shape 调试说明

这个 notebook 只做代码阅读、模块提取和 toy example 演示，不会修改项目已有文件。

我参考了 `outputs/ablation_all_new_modules/ablation_summary.md`、`engine/deim/hybrid_encoder.py`、`engine/deim/dfine_decoder.py`、`engine/deim/dental_prior.py` 和 `engine/deim/deim_criterion.py`。下面保留的是消融里更值得继续研究的模块：

- `CMFA / AdaptiveInterleaveFusion`：单模块 `abl08` AP 0.5393，相比 baseline +0.0049；和 HOOA 组合 `abl34` AP 0.5426，是旧分支里最强组合。
- `HOOA / HighOrderOmniAttention`：单独不强，但与 CMFA 协同明显，说明高阶局部纹理、上下文和边缘分支需要跨层融合提供稳定输入。
- `UCM-Uncertainty` 和 `MPF`：单项或组合有小幅收益，尤其 UCM 的 AP75 信号较好，适合边界定位优化。
- `SMWG-SpectralWavelet`：`abl87` AP 0.5368，+0.0024；在 phone tooth-prior 矩阵里 `SMWG+Prototype` 也有明显提升。
- `CED-v2`：单独不一定强，但 `CMFA+HOOA+CED-v2` 的 `abl72` AP 0.5423，和 best branch 牙齿先验矩阵有较好表现。
- `Tooth Prototype / Spatial-Shape Prior`：在 phone 与 DENTEX 分支上有选择性收益，和牙位结构强相关。
- `NWD loss`：单项 `abl39` 小幅正收益，适合密集小框的牙齿检测。

每个 forward 里我都加了 `verbose=True` 时的 `.shape` 打印，便于你在 notebook 中直接看张量如何变化。

In [1]:
# 读取消融表，快速筛出 delta > 0 的 done 实验。
# 这一步不依赖 pandas，避免 notebook 在轻量环境里因为缺包跑不起来。
from pathlib import Path

summary_path = Path('/home/wuxi/Dent-FINE/outputs/ablation_all_new_modules/ablation_summary.md')
rows = []
for line in summary_path.read_text(encoding='utf-8').splitlines():
    if not line.startswith('| abl'):
        continue
    parts = [p.strip() for p in line.strip('|').split('|')]
    if len(parts) < 9:
        continue
    exp, module, status, best_ap, ap50, ap75, delta, best_epoch, params = parts[:9]
    if status != 'done' or delta == '-':
        continue
    try:
        rows.append({
            'experiment': exp,
            'module': module,
            'best_ap': float(best_ap),
            'ap50': float(ap50),
            'ap75': float(ap75),
            'delta': float(delta.replace('+', '')),
            'params': params,
        })
    except ValueError:
        pass

positive = sorted([r for r in rows if r['delta'] > 0], key=lambda r: r['delta'], reverse=True)
print('Top positive rows from ablation_summary.md:')
for r in positive[:25]:
    print(f"{r['experiment']:45s} {r['module']:45s} AP={r['best_ap']:.4f} delta={r['delta']:+.4f} params={r['params']}")

Top positive rows from ablation_summary.md:
abl121_dentex_cmfa_hooa_cedv2_spatial_shape   DENTEX-CMFA+HOOA+CED-v2-SpatialShape          AP=0.5445 delta=+0.0088 params=16.54M
abl34_cmfa_hooa                               CMFA+HOOA                                     AP=0.5426 delta=+0.0083 params=12.45M
abl129_dentex_cmfa_hooa_spatial_shape         DENTEX-CMFA+HOOA-SpatialShape                 AP=0.5415 delta=+0.0082 params=12.47M
abl72_cmfa_hooa_ced_proj                      CMFA+HOOA+CED-v2                              AP=0.5423 delta=+0.0080 params=16.52M
abl108_phone_prototype                        Phone-Prototype                               AP=0.5424 delta=+0.0070 params=15.62M
abl155_coco_dualopt_adamw                     DualOpt-AdamW                                 AP=0.5412 delta=+0.0068 params=10.63M
abl140_phone_smwg_prototype                   Phone-SMWG-SpectralWavelet-Prototype          AP=0.5350 delta=+0.0067 params=11.28M
abl113_dentex_spatial_shape                   

## 公共工具

项目源码里大量使用 `ConvNormLayer_fuse` 和 `get_activation`。为了让 notebook 能独立运行，下面用一个等价的轻量 `ConvBNAct` 代替。真实项目中的模块定义仍然在 `engine/deim/hybrid_encoder.py`。

In [2]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

def get_activation(name='silu'):
    # 和项目里的 get_activation 作用一致：根据字符串返回激活层。
    name = (name or '').lower()
    if name in ('silu', 'swish'):
        return nn.SiLU(inplace=True)
    if name == 'gelu':
        return nn.GELU()
    if name == 'relu':
        return nn.ReLU(inplace=True)
    return nn.Identity()

class ConvBNAct(nn.Module):
    # notebook 版 ConvNormLayer_fuse：Conv2d + BN + Act。
    # 真实项目里 ConvNormLayer_fuse 还支持部署阶段 BN 融合，这里只保留理解模块所需的行为。
    def __init__(self, ch_in, ch_out, kernel_size=1, stride=1, groups=1, bias=False, act='silu'):
        super().__init__()
        padding = (kernel_size - 1) // 2
        self.conv = nn.Conv2d(ch_in, ch_out, kernel_size, stride, padding, groups=groups, bias=bias)
        self.bn = nn.BatchNorm2d(ch_out)
        self.act = get_activation(act)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

def log_shape(name, x, verbose=True):
    # 统一的 shape 打印函数；列表特征会打印每一层 FPN feature map 的形状。
    if not verbose:
        return
    if isinstance(x, (list, tuple)):
        print(f'{name}: {[tuple(t.shape) for t in x]}')
    elif torch.is_tensor(x):
        print(f'{name}: {tuple(x.shape)}')
    else:
        print(f'{name}: {x}')

## 1. CMFA: AdaptiveInterleaveFusion

源码位置：`engine/deim/hybrid_encoder.py::AdaptiveInterleaveFusion`。

它出现在 FPN top-down 和 PAN bottom-up 的跨层融合位置。普通 FPN 是直接 `concat([高层上采样, 低层细节])`，CMFA 先用通道门控让两路特征互相注入，再用空间门控抑制不可靠位置。

为什么对牙齿检测有用：牙齿目标密集、边界细、相邻类别很像。高层语义知道“这里像牙齿/牙弓”，低层特征保留牙冠边缘和缝隙。CMFA 的价值就是让这两路信息在进入 FPN/PAN block 之前先互相校正，所以 `AP50` 提升明显，也能给后续 HOOA 提供更干净的输入。

In [ ]:
class AdaptiveInterleaveFusion(nn.Module):
    def __init__(self, channels, fuse_op='cat', reduction=4, act='silu'):
        super().__init__()
        hidden = max((channels * 2) // reduction, 16)
        self.channels = channels
        self.fuse_op = fuse_op

        # 通道门控：输入是两路拼接特征 [B, 2C, H, W]，输出两组 gate，各自控制一路特征。
        self.channel_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels * 2, hidden, kernel_size=1, bias=False),
            get_activation(act),
            nn.Conv2d(hidden, channels * 2, kernel_size=1, bias=False),
            nn.Sigmoid(),
        )

        # 空间门控：用 mean/max 两张统计图判断哪些空间位置更可信。
        self.spatial_gate = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False),
            nn.Sigmoid(),
        )

        # 项目里 fuse_op='cat' 时输出 [B, 2C, H, W]，后面 FPN/PAN block 再降回 C。
        # 如果改成 fuse_op='sum'，这里会用 1x1 把 [2C] 投影回 [C]。
        self.out_proj = ConvBNAct(channels * 2, channels, 1, 1, act=act) if fuse_op == 'sum' else nn.Identity()

    def forward(self, feat_a, feat_b, verbose=True):
        log_shape('CMFA/feat_a', feat_a, verbose)
        log_shape('CMFA/feat_b', feat_b, verbose)

        fused0 = torch.cat([feat_a, feat_b], dim=1)
        log_shape('CMFA/concat before gate', fused0, verbose)

        gate_a, gate_b = self.channel_gate(fused0).split(self.channels, dim=1)
        log_shape('CMFA/gate_a', gate_a, verbose)
        log_shape('CMFA/gate_b', gate_b, verbose)

        feat_a_gated = feat_a * gate_a
        feat_b_gated = feat_b * gate_b

        # 互注入：a 保留自身，同时加入 b 的 gated 信息；b 也对称加入 a 的 gated 信息。
        fused = torch.cat([feat_a + feat_b_gated, feat_b + feat_a_gated], dim=1)
        log_shape('CMFA/interleaved fused', fused, verbose)

        spatial_stats = torch.cat([
            fused.mean(dim=1, keepdim=True),
            fused.max(dim=1, keepdim=True).values,
        ], dim=1)
        log_shape('CMFA/spatial stats', spatial_stats, verbose)

        fused = fused * self.spatial_gate(spatial_stats)
        out = self.out_proj(fused)
        log_shape('CMFA/out', out, verbose)
        return out

# Toy example：模拟 FPN 中高层上采样特征和低层特征。
cmfa = AdaptiveInterleaveFusion(channels=64, fuse_op='cat').eval()
feat_high_up = torch.randn(2, 64, 40, 40)
feat_low = torch.randn(2, 64, 40, 40)
with torch.no_grad():
    cmfa_out = cmfa(feat_high_up, feat_low, verbose=True)

## 2. HOOA: HighOrderOmniAttention

源码位置：`engine/deim/hybrid_encoder.py::HighOrderOmniAttention`。

HOOA 把局部纹理、大范围上下文、高频边缘、通道注意力和空间注意力合在一个轻量残差里。它单独使用时可能会放大噪声或错误上下文，但和 CMFA 配合后，输入特征已经被跨层语义校正，因此表现明显变好。

为什么对项目有用：全景牙片和口内手机图里，单颗牙的轮廓、牙缝、牙根方向都是局部高频信号；上下颌、牙弓排列又是大范围结构。HOOA 的 `local/context/edge/order2` 刚好覆盖这两类信息。

In [ ]:
class HighOrderOmniAttention(nn.Module):
    def __init__(self, channels, act='silu'):
        super().__init__()
        hidden = max(channels // 8, 16)

        # 局部分支：depthwise 3x3 捕捉牙冠边缘、牙缝等局部纹理。
        self.local_branch = nn.Sequential(
            ConvBNAct(channels, channels, 3, 1, groups=channels, act=act),
            ConvBNAct(channels, channels, 1, 1, act=act),
        )

        # 上下文分支：大 dilation 的 7x7 depthwise 卷积扩大感受野，看牙弓和邻牙关系。
        self.context_branch = nn.Sequential(
            nn.Conv2d(channels, channels, 7, padding=9, dilation=3, groups=channels, bias=False),
            nn.BatchNorm2d(channels),
            get_activation(act),
            nn.Conv2d(channels, channels, 1, bias=False),
            nn.BatchNorm2d(channels),
        )

        # 边缘分支：x - avg_pool(x) 得到高频残差，再做轻量卷积。
        self.edge_branch = nn.Sequential(
            ConvBNAct(channels, channels, 3, 1, groups=channels, act=act),
            ConvBNAct(channels, channels, 1, 1, act=act),
        )

        self.channel_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, 1, bias=False),
            get_activation(act),
            nn.Conv2d(hidden, channels, 1, bias=True),
            nn.Sigmoid(),
        )
        self.spatial_gate = nn.Sequential(nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False), nn.Sigmoid())
        self.out_proj = ConvBNAct(channels * 3, channels, 1, 1, act=act)

        # 负初始化让模块初期只是很小的残差，不会破坏 baseline。
        self.res_scale = nn.Parameter(torch.tensor(-2.5))

    def forward(self, x, verbose=True):
        log_shape('HOOA/x', x, verbose)
        local = self.local_branch(x)
        context = self.context_branch(x)
        high = x - F.avg_pool2d(x, kernel_size=3, stride=1, padding=1)
        edge = self.edge_branch(high)
        log_shape('HOOA/local', local, verbose)
        log_shape('HOOA/context', context, verbose)
        log_shape('HOOA/high-frequency residual', high, verbose)
        log_shape('HOOA/edge', edge, verbose)

        # 二阶交互：局部纹理乘以上下文门控，相当于“上下文确认过的局部细节”。
        order2 = local * torch.sigmoid(context)
        fused = self.out_proj(torch.cat([local, order2, edge], dim=1))
        log_shape('HOOA/fused before gates', fused, verbose)

        spatial_stats = torch.cat([fused.mean(dim=1, keepdim=True), fused.max(dim=1, keepdim=True).values], dim=1)
        fused = fused * self.channel_gate(fused) * self.spatial_gate(spatial_stats)
        alpha = 0.30 * torch.sigmoid(self.res_scale)
        out = x + alpha * fused
        log_shape('HOOA/out', out, verbose)
        return out

hooa = HighOrderOmniAttention(channels=64).eval()
x = torch.randn(2, 64, 40, 40)
with torch.no_grad():
    hooa_out = hooa(x, verbose=True)

## 3. UCM 与 MPF

源码位置：`UncertaintyContextRefiner` 和 `MultiPathFeatureFusion`。

UCM 根据局部方差估计不确定区域：方差高的位置更依赖 detail，方差低的位置更依赖平滑 context。MPF 则并行使用空间、通道和上下文路径。它们的收益小于 CMFA/HOOA，但对 AP75 和边界质量有价值，适合做轻量补充或和 CMFA 做拆分实验。

In [ ]:
class UncertaintyContextRefiner(nn.Module):
    def __init__(self, channels, act='silu'):
        super().__init__()
        hidden = max(channels // 8, 16)
        self.context_proj = ConvBNAct(channels, channels, 1, 1, act=act)
        self.detail_proj = nn.Sequential(
            ConvBNAct(channels, channels, 3, 1, groups=channels, act=act),
            ConvBNAct(channels, channels, 1, 1, act=act),
        )
        self.uncertainty = nn.Sequential(
            nn.Conv2d(channels * 2, hidden, 1, bias=False),
            nn.BatchNorm2d(hidden),
            get_activation(act),
            nn.Conv2d(hidden, 1, 1, bias=True),
            nn.Sigmoid(),
        )
        self.res_scale = nn.Parameter(torch.tensor(-2.0))

    def forward(self, x, verbose=True):
        log_shape('UCM/x', x, verbose)
        local_mean = F.avg_pool2d(x, kernel_size=5, stride=1, padding=2)
        local_var = F.avg_pool2d(x.pow(2), kernel_size=5, stride=1, padding=2) - local_mean.pow(2)
        local_var = local_var.relu()
        uncertainty = self.uncertainty(torch.cat([x, local_var], dim=1))
        context = self.context_proj(local_mean)
        detail = self.detail_proj(x - local_mean)
        log_shape('UCM/local_mean', local_mean, verbose)
        log_shape('UCM/local_var', local_var, verbose)
        log_shape('UCM/uncertainty mask', uncertainty, verbose)
        update = uncertainty * detail + (1.0 - uncertainty) * context
        alpha = 0.35 * torch.sigmoid(self.res_scale)
        out = x + alpha * update
        log_shape('UCM/out', out, verbose)
        return out

class MultiPathFeatureFusion(nn.Module):
    def __init__(self, in_channels=256, reduction=16):
        super().__init__()
        hidden = max(in_channels // reduction, 8)
        self.spatial_path = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=5, padding=2, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.Conv2d(in_channels, in_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.GELU(),
        )
        self.channel_path = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, hidden, 1, bias=False),
            nn.GELU(),
            nn.Conv2d(hidden, in_channels, 1, bias=False),
            nn.Sigmoid(),
        )
        self.context_path = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 2, 1, bias=False),
            nn.BatchNorm2d(in_channels // 2),
            nn.GELU(),
            nn.Conv2d(in_channels // 2, in_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(in_channels),
        )
        self.fusion_weights = nn.Parameter(torch.ones(3) / 3)
        self.res_scale = nn.Parameter(torch.tensor(-2.0))

    def forward(self, x, verbose=True):
        log_shape('MPF/x', x, verbose)
        spatial = self.spatial_path(x)
        channel_feat = x * self.channel_path(x)
        context = self.context_path(x)
        log_shape('MPF/spatial', spatial, verbose)
        log_shape('MPF/channel_feat', channel_feat, verbose)
        log_shape('MPF/context', context, verbose)
        weights = F.softmax(self.fusion_weights, dim=0)
        fused = weights[0] * spatial + weights[1] * channel_feat + weights[2] * context
        alpha = 0.35 * torch.sigmoid(self.res_scale)
        out = x + alpha * fused
        log_shape('MPF/out', out, verbose)
        return out

ucm = UncertaintyContextRefiner(64).eval()
mpf = MultiPathFeatureFusion(64).eval()
x = torch.randn(2, 64, 40, 40)
with torch.no_grad():
    _ = ucm(x, verbose=True)
    _ = mpf(x, verbose=True)

## 4. SMWG-SpectralWavelet

源码位置：`SMWGFourierSpectralModulation` 与 `SMWGSpectralWaveletRefiner`。

SMWG 先对每个 FPN level 做 Fourier 频带调制，再把高分辨率层的高通细节引导到更粗层。牙齿检测里，牙缝、牙冠边缘、根尖边界是典型高频结构；但只增强高频容易带噪，所以源码使用 residual scale 和 channel gate 控制更新幅度。

In [ ]:
class SMWGFourierSpectralModulation(nn.Module):
    def __init__(self, channels, res_init=-2.0, act='silu'):
        super().__init__()
        self.pre = ConvBNAct(channels, channels, 1, 1, act=act)
        self.post = ConvBNAct(channels, channels, 1, 1, act=act)
        self.band_logits = nn.Parameter(torch.zeros(channels, 3))
        self.channel_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, max(channels // 8, 16), 1),
            nn.GELU(),
            nn.Conv2d(max(channels // 8, 16), channels, 1),
            nn.Sigmoid(),
        )
        self.res_scale = nn.Parameter(torch.tensor(float(res_init)))

    @staticmethod
    def _radial_bands(h, wf, device, dtype):
        # 构造低/中/高三个径向频带；rfftfreq 对应 rfft2 的最后一维。
        fy = torch.fft.fftfreq(h, device=device, dtype=dtype).view(h, 1)
        fx = torch.fft.rfftfreq((wf - 1) * 2, device=device, dtype=dtype).view(1, wf)
        r = torch.sqrt(fx.square() + fy.square())
        low = torch.exp(-(r / 0.16).square())
        mid = torch.exp(-((r - 0.28) / 0.16).square())
        high = (1.0 - torch.exp(-(r / 0.34).square())).clamp(0, 1)
        return torch.stack([low, mid, high], dim=0)

    def forward(self, x, verbose=True):
        log_shape('SMWG-Fourier/x', x, verbose)
        z = self.pre(x)
        b, c, h, w = z.shape
        spec = torch.fft.rfft2(z.float(), norm='ortho')
        log_shape('SMWG-Fourier/spec(real view omitted)', torch.view_as_real(spec), verbose)
        bands = self._radial_bands(h, spec.shape[-1], z.device, z.float().dtype)
        log_shape('SMWG-Fourier/radial bands', bands, verbose)
        band_weights = torch.softmax(self.band_logits.float(), dim=-1)
        modulation = torch.einsum('ck,khw->chw', band_weights, bands).unsqueeze(0)
        spec = spec * (1.0 + 0.20 * torch.tanh(modulation))
        filtered = torch.fft.irfft2(spec, s=(h, w), norm='ortho').to(dtype=x.dtype)
        update = self.post(filtered) * self.channel_gate(z)
        alpha = 0.35 * torch.sigmoid(self.res_scale)
        out = x + alpha * update
        log_shape('SMWG-Fourier/out', out, verbose)
        return out

class SMWGSpectralWaveletRefiner(nn.Module):
    def __init__(self, channels, num_levels=3, res_init=-1.8, act='silu'):
        super().__init__()
        self.num_levels = int(num_levels)
        self.spectral = nn.ModuleList([
            SMWGFourierSpectralModulation(channels, res_init=res_init, act=act)
            for _ in range(self.num_levels)
        ])
        self.detail_proj = nn.ModuleList([ConvBNAct(channels, channels, 1, 1, act=act) for _ in range(self.num_levels - 1)])
        self.coarse_fuse = nn.ModuleList([ConvBNAct(channels, channels, 3, 1, groups=channels, act=act) for _ in range(self.num_levels - 1)])
        self.res_scale = nn.Parameter(torch.tensor(float(res_init)))

    @staticmethod
    def _high_pass(x):
        # notebook 版 fallback：用均值池化近似低频，高频 = 原图 - 平滑图。
        return x - F.avg_pool2d(x, kernel_size=3, stride=1, padding=1)

    def forward(self, feats, verbose=True):
        log_shape('SMWG/input feats', feats, verbose)
        feats = [self.spectral[i](x, verbose=verbose) for i, x in enumerate(feats)]
        outs = list(feats)
        alpha = 0.35 * torch.sigmoid(self.res_scale)
        for i in range(len(feats) - 1):
            fine_detail = self._high_pass(feats[i])
            log_shape(f'SMWG/level {i} fine_detail before resize', fine_detail, verbose)
            fine_detail = F.interpolate(fine_detail, size=feats[i + 1].shape[2:], mode='bilinear', align_corners=False)
            guide = torch.sigmoid(self.detail_proj[i](fine_detail))
            update = self.coarse_fuse[i](feats[i + 1] * guide + fine_detail)
            outs[i + 1] = outs[i + 1] + alpha * update
            log_shape(f'SMWG/level {i + 1} updated', outs[i + 1], verbose)
        log_shape('SMWG/output feats', outs, verbose)
        return outs

smwg = SMWGSpectralWaveletRefiner(channels=32, num_levels=3).eval()
feats = [torch.randn(2, 32, 64, 64), torch.randn(2, 32, 32, 32), torch.randn(2, 32, 16, 16)]
with torch.no_grad():
    smwg_outs = smwg(feats, verbose=True)

## 5. CED-v2: Cascade Encoder-Decoder

源码位置：`CEDCascadeStage` 与 `CEDCascadeEncoderDecoder`。

真实项目中 CED stage 的融合块使用 `RepNCSPELAN4` 或 `CSPLayer`。为了 notebook 能独立运行，下面用 `SimpleFusionBlock` 代替那两个重块，但保留 CED 的核心数据流：先 top-down 注入高层语义，再 bottom-up 回传低层定位细节，最后用残差比例和原特征相加。

为什么有用：牙齿检测既需要粗粒度牙弓/上下颌语义，也需要细粒度牙缝和边界。CED 的两遍级联让语义和细节反复交流。消融里它单独不是强项，但在 `CMFA+HOOA` 后作为级联补充有明显价值。

In [ ]:
class SimpleFusionBlock(nn.Module):
    # notebook 替代版：把 [B, 2C, H, W] 融合成 [B, C, H, W]。
    def __init__(self, in_channels, out_channels, act='silu'):
        super().__init__()
        self.block = nn.Sequential(
            ConvBNAct(in_channels, out_channels, 1, 1, act=act),
            ConvBNAct(out_channels, out_channels, 3, 1, groups=out_channels, act=act),
            ConvBNAct(out_channels, out_channels, 1, 1, act=act),
        )

    def forward(self, x):
        return self.block(x)

class CEDCascadeStage(nn.Module):
    def __init__(self, channels, num_levels=3, res_init=-1.6, act='silu'):
        super().__init__()
        self.num_levels = int(num_levels)
        self.top_laterals = nn.ModuleList([ConvBNAct(channels, channels, 1, 1, act=act) for _ in range(num_levels - 1)])
        self.top_guides = nn.ModuleList([nn.Sequential(nn.Conv2d(channels, channels, 1), nn.Sigmoid()) for _ in range(num_levels - 1)])
        self.top_blocks = nn.ModuleList([SimpleFusionBlock(channels * 2, channels, act=act) for _ in range(num_levels - 1)])
        self.bottom_downs = nn.ModuleList([ConvBNAct(channels, channels, 3, 2, act=act) for _ in range(num_levels - 1)])
        self.bottom_guides = nn.ModuleList([nn.Sequential(nn.Conv2d(channels, channels, 1), nn.Sigmoid()) for _ in range(num_levels - 1)])
        self.bottom_blocks = nn.ModuleList([SimpleFusionBlock(channels * 2, channels, act=act) for _ in range(num_levels - 1)])
        self.stage_scale = nn.Parameter(torch.tensor(float(res_init)))

    def forward(self, feats, verbose=True):
        # feats 顺序和 FPN 输出一致：从高分辨率到低分辨率，例如 P3, P4, P5。
        log_shape('CED/input feats', feats, verbose)

        # Top-down：最粗层开始，逐级上采样，引导更细层。
        inner = [feats[-1]]
        for rev_idx in range(self.num_levels - 1, 0, -1):
            pos = self.num_levels - 1 - rev_idx
            high = self.top_laterals[pos](inner[0])
            low = feats[rev_idx - 1]
            high_up = F.interpolate(high, size=low.shape[2:], mode='bilinear', align_corners=False)
            guide = self.top_guides[pos](high_up)
            fused = torch.cat([high_up, low * (1.0 + guide)], dim=1)
            log_shape(f'CED/top fused pos={pos}', fused, verbose)
            inner.insert(0, self.top_blocks[pos](fused))

        # Bottom-up：从最细层往粗层回传定位和边缘信息。
        outs = [inner[0]]
        for idx in range(self.num_levels - 1):
            low = outs[-1]
            high = inner[idx + 1]
            low_down = self.bottom_downs[idx](low)
            if low_down.shape[2:] != high.shape[2:]:
                low_down = F.interpolate(low_down, size=high.shape[2:], mode='bilinear', align_corners=False)
            guide = self.bottom_guides[idx](high)
            fused = torch.cat([low_down, high * (1.0 + guide)], dim=1)
            log_shape(f'CED/bottom fused idx={idx}', fused, verbose)
            outs.append(self.bottom_blocks[idx](fused))

        alpha = 0.35 * torch.sigmoid(self.stage_scale)
        outs = [old + alpha * (new - old) for old, new in zip(feats, outs)]
        log_shape('CED/output feats', outs, verbose)
        return outs

class CEDCascadeEncoderDecoder(nn.Module):
    def __init__(self, channels, num_levels=3, stages=2, res_init=-1.6, act='silu'):
        super().__init__()
        self.stages = nn.ModuleList([CEDCascadeStage(channels, num_levels, res_init, act) for _ in range(stages)])

    def forward(self, feats, verbose=True):
        for i, stage in enumerate(self.stages):
            print(f'-- CED stage {i} --') if verbose else None
            feats = stage(feats, verbose=verbose)
        return feats

ced = CEDCascadeEncoderDecoder(channels=32, num_levels=3, stages=2).eval()
feats = [torch.randn(2, 32, 64, 64), torch.randn(2, 32, 32, 32), torch.randn(2, 32, 16, 16)]
with torch.no_grad():
    ced_outs = ced(feats, verbose=True)

## 6. 牙齿 Prototype 与 Spatial/Shape Prior

源码位置：`dfine_decoder.py::_prototype_logits`、`ToothTemplateLogitPrior`，以及 `dental_prior.py::spatial_template_score / shape_template_score`。

这类模块不是通用 COCO 检测技巧，而是利用 FDI 牙位结构：每个类别天然有上下颌位置、左右顺序、宽高形状和外观原型。消融中 prototype 和 spatial-shape 在不同数据集/分支上表现不同：phone 图像里 prototype 更有信号，DENTEX 和强分支里 spatial-shape 更有信号。实际使用时建议按数据集分开保留开关，不要默认全开。

In [ ]:
def fdi52_order():
    # 32 颗恒牙 + 20 颗乳牙的常见 FDI 顺序，和项目里 FDI 先验思路一致。
    order = []
    for quadrant in [1, 2, 3, 4]:
        for tooth in range(1, 9):
            order.append(quadrant * 10 + tooth)
    for quadrant in [5, 6, 7, 8]:
        for tooth in range(1, 6):
            order.append(quadrant * 10 + tooth)
    return order

def fdi_arch_sequence(is_upper=True, dentition='perm'):
    # 用一个从左到右的牙弓顺序生成空间中心；这里只用于 notebook 演示。
    if dentition == 'perm' and is_upper:
        return list(range(18, 10, -1)) + list(range(21, 29))
    if dentition == 'perm' and not is_upper:
        return list(range(48, 40, -1)) + list(range(31, 39))
    if dentition == 'primary' and is_upper:
        return list(range(55, 50, -1)) + list(range(61, 66))
    return list(range(85, 80, -1)) + list(range(71, 76))

def make_fdi_centers(num_classes=52, device='cpu', dtype=torch.float32):
    order = fdi52_order()[:num_classes]
    centers = torch.full((num_classes, 2), 0.5, device=device, dtype=dtype)
    for cid, fdi in enumerate(order):
        q = fdi // 10
        is_upper = q in (1, 2, 5, 6)
        dentition = 'primary' if fdi >= 50 else 'perm'
        seq = fdi_arch_sequence(is_upper=is_upper, dentition=dentition)
        x = (seq.index(fdi) + 0.5) / len(seq) if fdi in seq else 0.5
        y = 0.36 if is_upper else 0.64
        centers[cid] = torch.tensor([x, y], device=device, dtype=dtype)
    return centers

class ToothPrototypeHead(nn.Module):
    # 对应 dfine_decoder.py 里的 _prototype_logits：query 和每类 K 个原型做 cosine similarity。
    def __init__(self, num_classes=52, dim=256, prototype_count=4, tau=0.07):
        super().__init__()
        self.num_classes = num_classes
        self.prototype_count = prototype_count
        self.tau = tau
        self.tooth_prototypes = nn.Parameter(torch.randn(num_classes, prototype_count, dim) * 0.02)

    def forward(self, query_feats, verbose=True):
        log_shape('Prototype/query_feats', query_feats, verbose)
        feats = F.normalize(query_feats, dim=-1)
        protos = F.normalize(self.tooth_prototypes, dim=-1)
        sim = torch.einsum('bqd,ckd->bqck', feats, protos)
        log_shape('Prototype/sim [B,Q,C,K]', sim, verbose)
        logits = torch.logsumexp(sim / max(self.tau, 1e-6), dim=-1)
        log_shape('Prototype/logits [B,Q,C]', logits, verbose)
        return logits

class ToothTemplateLogitPrior(nn.Module):
    # 提取自 dfine_decoder.py 的 ToothTemplateLogitPrior，并做 notebook 自包含改写。
    # 它把 query 外观模板相似度和参考点空间先验叠加到分类 logits 上。
    def __init__(self, num_classes=52, dim=256, num_templates=4, template_scale=0.08, spatial_scale=0.05, spatial_sigma=0.18, temperature=0.20):
        super().__init__()
        self.num_classes = int(num_classes)
        self.dim = int(dim)
        self.num_templates = max(int(num_templates), 1)
        self.template_scale = float(template_scale)
        self.spatial_scale = float(spatial_scale)
        self.spatial_sigma = float(spatial_sigma)
        self.temperature = max(float(temperature), 1e-4)
        self.templates = nn.Parameter(torch.randn(self.num_classes, self.num_templates, self.dim) * 0.02)
        self.query_proj = nn.Linear(self.dim, self.dim, bias=False)
        self.template_proj = nn.Linear(self.dim, self.dim, bias=False)
        self.mix_gate = nn.Sequential(
            nn.LayerNorm(self.dim), nn.Linear(self.dim, max(self.dim // 4, 16)), nn.GELU(),
            nn.Linear(max(self.dim // 4, 16), 1), nn.Sigmoid(),
        )

    @staticmethod
    def _normalize_prior(prior):
        prior = prior - prior.mean(dim=-1, keepdim=True)
        return prior / prior.std(dim=-1, keepdim=True).clamp_min(1e-3)

    def forward(self, scores, query_feats, ref_points=None, verbose=True):
        log_shape('TemplatePrior/scores [B,Q,C]', scores, verbose)
        log_shape('TemplatePrior/query_feats [B,Q,D]', query_feats, verbose)
        if ref_points is not None:
            log_shape('TemplatePrior/ref_points [B,Q,2]', ref_points, verbose)

        q = F.normalize(self.query_proj(query_feats.float()), dim=-1)
        templates = F.normalize(self.template_proj(self.templates.float()), dim=-1)
        sim = torch.einsum('bqd,ckd->bqck', q, templates)
        sim = torch.logsumexp(sim / self.temperature, dim=-1) * self.temperature
        update = self.template_scale * self._normalize_prior(sim)
        log_shape('TemplatePrior/template update [B,Q,C]', update, verbose)

        if ref_points is not None and self.spatial_scale > 0:
            ref = torch.nan_to_num(ref_points[..., :2].float(), nan=0.5, posinf=1.0, neginf=0.0).clamp(0, 1)
            centers = make_fdi_centers(self.num_classes, device=scores.device, dtype=ref.dtype)
            dist = torch.cdist(ref, centers.view(1, self.num_classes, 2).expand(ref.shape[0], -1, -1), p=2)
            spatial = -dist.pow(2) / (2.0 * max(self.spatial_sigma, 1e-4) ** 2)
            update = update + self.spatial_scale * self._normalize_prior(spatial)
            log_shape('TemplatePrior/spatial prior [B,Q,C]', spatial, verbose)

        gate = self.mix_gate(query_feats.float()).to(dtype=scores.dtype)
        log_shape('TemplatePrior/mix gate [B,Q,1]', gate, verbose)
        out = scores + gate * update.to(dtype=scores.dtype)
        log_shape('TemplatePrior/out scores [B,Q,C]', out, verbose)
        return out

def spatial_template_score(boxes_cxcywh, prior, clamp=20.0, verbose=True):
    # 提取自 dental_prior.py：用中心点的 Mahalanobis 距离给每个类别打空间分数。
    log_shape('SpatialShape/boxes', boxes_cxcywh, verbose)
    mu = prior['spatial_mu']
    inv_cov = prior['spatial_inv_cov']
    centers = boxes_cxcywh[..., :2]
    diff = centers[:, :, None, :] - mu[None, None, :, :]
    score = -0.5 * torch.einsum('bqci,cij,bqcj->bqc', diff, inv_cov, diff)
    score = score.clamp(min=-float(clamp), max=0.0)
    log_shape('SpatialShape/spatial score [B,Q,C]', score, verbose)
    return score

def shape_template_score(boxes_cxcywh, prior, clamp=20.0, verbose=True):
    # 提取自 dental_prior.py：用 log(w/h) 的 Mahalanobis 距离给每个类别打形状分数。
    mu = prior['shape_mu']
    inv_cov = prior['shape_inv_cov']
    wh = boxes_cxcywh[..., 2:].clamp_min(1e-5).log()
    diff = wh[:, :, None, :] - mu[None, None, :, :]
    score = -0.5 * torch.einsum('bqci,cij,bqcj->bqc', diff, inv_cov, diff)
    score = score.clamp(min=-float(clamp), max=0.0)
    log_shape('SpatialShape/shape score [B,Q,C]', score, verbose)
    return score

# Toy example：52 个 query 对应 52 类 FDI 牙位。
B, Q, C, D = 2, 52, 52, 64
scores = torch.randn(B, Q, C)
query_feats = torch.randn(B, Q, D)
ref_points = torch.rand(B, Q, 2)
boxes = torch.cat([ref_points, torch.rand(B, Q, 2) * 0.10 + 0.03], dim=-1)

prototype_head = ToothPrototypeHead(num_classes=C, dim=D, prototype_count=4).eval()
template_prior = ToothTemplateLogitPrior(num_classes=C, dim=D).eval()
prior = {
    'spatial_mu': make_fdi_centers(C),
    'spatial_inv_cov': torch.eye(2).unsqueeze(0).repeat(C, 1, 1) / (0.18 ** 2),
    'shape_mu': torch.zeros(C, 2),
    'shape_inv_cov': torch.eye(2).unsqueeze(0).repeat(C, 1, 1),
}

with torch.no_grad():
    proto_logits = prototype_head(query_feats, verbose=True)
    updated_scores = template_prior(scores, query_feats, ref_points, verbose=True)
    spatial_score = spatial_template_score(boxes, prior, verbose=True)
    shape_score = shape_template_score(boxes, prior, verbose=True)

## 7. NWD Loss

源码位置：`engine/deim/deim_criterion.py::loss_nwd`。

NWD 把框看成高斯分布，用中心距离和宽高距离组成 Wasserstein 距离，再转成相似度。牙齿框通常密集且偏小，小框的 IoU 对几像素偏移非常敏感；NWD 的梯度更平滑，所以消融里有小幅正收益。它不是替代 IoU/GIoU，而是适合作为辅助 loss。

In [ ]:
def normalized_wasserstein_loss(src_boxes, target_boxes, nwd_normalizer=0.08, verbose=True):
    # src_boxes / target_boxes: [N, 4]，格式是归一化 cx, cy, w, h。
    log_shape('NWD/src_boxes [N,4]', src_boxes, verbose)
    log_shape('NWD/target_boxes [N,4]', target_boxes, verbose)
    center_dist = (src_boxes[:, :2] - target_boxes[:, :2]).pow(2).sum(dim=-1)
    size_dist = (src_boxes[:, 2:] - target_boxes[:, 2:]).pow(2).sum(dim=-1) / 4.0
    wasserstein = torch.sqrt((center_dist + size_dist).clamp_min(1e-12))
    nwd = torch.exp(-wasserstein / max(float(nwd_normalizer), 1e-6))
    loss = (1.0 - nwd).mean()
    log_shape('NWD/center_dist [N]', center_dist, verbose)
    log_shape('NWD/size_dist [N]', size_dist, verbose)
    log_shape('NWD/nwd similarity [N]', nwd, verbose)
    print(f'NWD loss = {loss.item():.6f}') if verbose else None
    return loss

src = torch.tensor([[0.50, 0.50, 0.08, 0.12], [0.25, 0.62, 0.06, 0.10]])
tgt = torch.tensor([[0.52, 0.49, 0.08, 0.11], [0.23, 0.61, 0.07, 0.10]])
_ = normalized_wasserstein_loss(src, tgt, verbose=True)

## 推荐使用方式

从当前消融结果看，我会把模块分成三层：

1. 主线优先：`CMFA + HOOA`。这是当前最清晰的强组合，AP 提升最大，且参数规模仍可控。
2. 分支补充：`CMFA + HOOA + CED-v2` 或 `SMWG`。CED-v2 更像多级语义-细节级联，SMWG 更偏高频/边界增强。两者都要保留开关，不建议盲目全堆。
3. 牙齿先验：phone 数据优先试 `Prototype`，DENTEX 或更规范全景数据优先试 `SpatialShape`。这符合两类数据的差别：手机图外观域偏移大，prototype 能补外观聚类；DENTEX 位置和形状先验更稳定。

负收益明显的模块不建议默认启用，例如 `WavEnhance`、`DynamicRF`、`FDI-FactorPrior` 单支、过重的全量 paper module 组合等。它们可以保留为论文实验讨论，但不是下一轮提 AP 的首选。